# RSPY 675

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-675

## 1. Initialization

In [ ]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_eopf(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)

In [ ]:
from rs_client.rs_client import RsClient
rs_server_href = os.getenv("RSPY_WEBSITE")
rs_server_api_key = os.environ.get("RSPY_APIKEY")
generic_client = RsClient(rs_server_href, rs_server_api_key, OWNER_ID, None)
dpr_client = generic_client.get_dpr_client()

In [ ]:
import requests
response = requests.get(f"{dpr_client.href_service}/_mgmt/ping")
response

In [ ]:
# 1. Connect to your Dask Gateway cluster
from dask_gateway import Gateway, BasicAuth
import dask

gateway_address = dask_cluster_eopf.gateway.address
cluster_name = dask_cluster_eopf.name
username = os.environ["LOCAL_DASK_USERNAME"]
password = os.environ["LOCAL_DASK_PASSWORD"]

gateway = Gateway(address=gateway_address, auth=BasicAuth(username, password))
cluster = gateway.connect(cluster_name)

# 2. Get a Dask client
from distributed import Client
client = cluster.get_client()

In [ ]:
s3_config = {
    "key": os.environ["S3_ACCESSKEY"],
    "secret": os.environ["S3_SECRETKEY"],
    "client_kwargs": {
        "endpoint_url": os.environ["S3_ENDPOINT"],
        "region_name": os.environ["S3_REGION"],
    },
}

In [ ]:
info = client.scheduler_info()["workers"]

In [ ]:
# Shutdown the clusters
shutdown_dask_clusters(gateway, cluster_name)
# Close the python objects
close_dask_clusters()